In [1]:
import os, numpy as np, pandas as pd, cv2
import tensorflow as tf
import albumentations as A
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import kagglehub

# 1. Dataset
path_cls = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
METADATA_PATH = os.path.join(path_cls, 'HAM10000_metadata.csv')
IMG_DIR_1 = os.path.join(path_cls, 'HAM10000_images_part_1')
IMG_DIR_2 = os.path.join(path_cls, 'HAM10000_images_part_2')

df = pd.read_csv(METADATA_PATH)
def get_image_path(image_id):
    for img_dir in [IMG_DIR_1, IMG_DIR_2]:
        p = os.path.join(img_dir, f'{image_id}.jpg')
        if os.path.exists(p): return p
    return None
df['image_path'] = df['image_id'].apply(get_image_path)
df = df.dropna(subset=['image_path']).reset_index(drop=True)
df['label'] = df['dx']

LABELS = sorted(df['label'].unique())
label_to_idx = {l: i for i, l in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)


Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.


In [2]:
# 2. Той самий спліт
df['label_idx'] = df['label'].map(label_to_idx)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss.split(df, df['label_idx'], groups=df['lesion_id']))
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=42)
train_idx, val_idx = next(gss2.split(train_val_df, train_val_df['label_idx'], groups=train_val_df['lesion_id']))
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print(f'Val: {len(val_df)}, Test: {len(test_df)}')


Val: 1525, Test: 1527


In [3]:
# 3. Load model
from google.colab import drive
drive.mount('/content/drive')

# ===== ЗМІНИ ШЛЯХ НА СВІЙ! =====
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    def call(self, y_true, y_pred):
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true = y_true * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        focal_weight = tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_sum(focal_weight * cross_entropy, axis=-1)
    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'label_smoothing': self.label_smoothing})
        return config

MODEL_PATH = '/content/drive/MyDrive/model1.keras'  # ЗМІНИ!
best_model = tf.keras.models.load_model(MODEL_PATH, custom_objects={'FocalLoss': FocalLoss})
print('Model loaded')


Mounted at /content/drive
Model loaded


In [4]:
# 4. TTA
IMG_SIZE = 260
tta_transforms = [
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1.0)]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.VerticalFlip(p=1.0)]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.RandomRotate90(p=1.0)]),
]

def predict_tta(model, image_path, transforms):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    preds = []
    for t in transforms:
        aug = t(image=img)['image']
        aug = preprocess_input(aug.astype(np.float32))
        pred = model.predict(aug[np.newaxis, ...], verbose=0)[0]
        preds.append(pred)
    return np.mean(preds, axis=0)



In [5]:
# 5. Predictions
preds_val = []
for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc='Val'):
    preds_val.append(predict_tta(best_model, row['image_path'], tta_transforms))
preds_val = np.array(preds_val)

preds_test = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test'):
    preds_test.append(predict_tta(best_model, row['image_path'], tta_transforms))
preds_test = np.array(preds_test)

y_val = val_df['label'].map(label_to_idx).values
y_test = test_df['label'].map(label_to_idx).values

Val:   0%|          | 0/1525 [00:00<?, ?it/s]

Test:   0%|          | 0/1527 [00:00<?, ?it/s]

In [6]:
# 6. Save
SAVE_DIR = '/content/drive/MyDrive/HAM10000_v4/'
os.makedirs(SAVE_DIR, exist_ok=True)
np.save(os.path.join(SAVE_DIR, 'preds_val_v4.npy'), preds_val)
np.save(os.path.join(SAVE_DIR, 'preds_test_v4.npy'), preds_test)
np.save(os.path.join(SAVE_DIR, 'y_val_v4.npy'), y_val)
np.save(os.path.join(SAVE_DIR, 'y_test_v4.npy'), y_test)

print(f'Val: {preds_val.shape}, Test: {preds_test.shape}')
print('Saved!')

Val: (1525, 7), Test: (1527, 7)
Saved!
